# 01 · Build the feature table

Downloads the Kaggle BTC/ETH 1-second order-book data and builds one table with PCA, raw-LOB, market and classic-factor features, labels and the train/val/test split. All model notebooks read this file.

| Group | # | Content |
|---|---|---|
| LOB | 60 | bid/ask distance + notional, 15 levels |
| PCA | ≤10 | StandardScaler + PCA on LOB, fitted on train, Kaiser criterion |
| Market | 15 | BTC/ETH spread, imbalance, volatility, momentum, cross-asset |
| Classic | 13 | queue imbalance, OFI, Kyle λ, depth ratio, tick rule, Amihud |

All features get a backward-looking 1-hour rolling z-score (no look-ahead).

In [ ]:
# ── Setup: works on Colab (clones the repo) and locally (run from notebooks/) ──
import os, sys

REPO_URL = "https://github.com/<your-username>/mat629-lob-prediction.git"   # TODO: your repo

if "google.colab" in sys.modules:
    if not os.path.exists("mat629-lob-prediction"):
        !git clone -q $REPO_URL
    %cd mat629-lob-prediction
    !pip install -q kagglehub pyarrow
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

# Where the feature table lives. On Colab you can point this at Google Drive
# (after drive.mount) so it survives runtime restarts.
FEATURES_PATH = "data/features.parquet"

In [ ]:
from lob.features import build_dataset

# DATA_DIR = None downloads with kagglehub; or point it at a folder with
# BTC_1sec.csv and ETH_1sec.csv. Set NROWS (e.g. 100_000) for a quick test.
DATA_DIR, NROWS = None, None
df, meta = build_dataset(DATA_DIR, NROWS, out_path=FEATURES_PATH)

## Diagnostics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from lob.features import CLASSIC_COLS, LOB_COLS, MARKET_COLS

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
evr = np.array(meta["pca"]["explained_variance_ratio"])
k = meta["pca"]["n_components_kept"]
ax[0].bar(range(1, len(evr) + 1), evr, color=["C0"] * k + ["lightgray"] * (len(evr) - k))
ax[0].plot(range(1, len(evr) + 1), evr.cumsum(), "o-", color="C3", ms=3)
ax[0].set_title(f"PCA explained variance (kept {k})")

t = df[df["is_target"]]
t.groupby("split")["label"].value_counts(normalize=True).unstack().loc[["train", "val", "test"]] \
    .plot.bar(ax=ax[1], rot=0, title="Label distribution by split")

stds = {g: df[cols].std() for g, cols in
        [("LOB", LOB_COLS), ("Market", MARKET_COLS), ("Classic", CLASSIC_COLS)]}
ax[2].boxplot(list(stds.values()), tick_labels=list(stds.keys()))
ax[2].axhline(1, ls=":", color="gray"); ax[2].set_title("Per-feature std after rolling z-score")
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()